In [1]:
import json
import os

from pathlib import Path
from google import genai

In [8]:
GEMINI_API_KEY = "AQ.Ab8RN6L3hgm9f4B0K1IOoxMgrUXKrBUW5v_VEY4LacX0Sr0C4w"

client = genai.Client(api_key=GEMINI_API_KEY)

In [3]:
#Load Tree JSON
import json

with open(
    "LuatLaoDong2019_tree.json",
    "r",
    encoding="utf-8"
) as f:
    document = json.load(f)

nodes = document["nodes"]

node_index = {
    node["node_id"]: node
    for node in nodes
}

print(f"Loaded {len(nodes)} nodes.")
print(f"Indexed {len(node_index)} nodes.")

Loaded 1169 nodes.
Indexed 1169 nodes.


In [4]:
query = "Nam gới được nghỉ hưu bao nhiêu tuổi?"

In [5]:
def create_node_selector_prompt(query, nodes):

    tree_json = json.dumps(
        nodes,
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
Bạn là hệ thống tìm kiếm và định vị thông tin trong Bộ luật Lao động Việt Nam 2019.

NHIỆM VỤ:
- Dựa trên TREE JSON được cung cấp, hãy tìm các node chứa thông tin liên quan trực tiếp hoặc cần thiết để trả lời câu hỏi của người dùng.
- Bạn KHÔNG được trả lời câu hỏi.
- Bạn chỉ thực hiện việc tìm kiếm và lựa chọn node phù hợp.

QUY TẮC TÌM KIẾM:
1. Chọn từ 0 đến 5 node phù hợp nhất.
2. Ưu tiên node cụ thể nhất có chứa thông tin cần thiết:
   - Nếu Điểm chứa đủ thông tin → chọn Điểm.
   - Nếu Khoản chứa đủ thông tin → chọn Khoản.
   - Nếu Điều chứa đủ thông tin → chọn Điều.
   - Chỉ chọn node cha khi node con không đủ thông tin hoặc
     cần node cha để hiểu ngữ cảnh.
3. Có thể chọn nhiều node nếu câu hỏi liên quan đến nhiều quy định
   khác nhau.
4. Không chọn node chỉ vì có từ khóa giống với câu hỏi.
   Phải xem xét ý nghĩa và nội dung của node.
5. Ưu tiên:
   - nội dung pháp luật trực tiếp trả lời câu hỏi;
   - node có phạm vi áp dụng phù hợp;
   - node quy định về điều kiện, quyền, nghĩa vụ, mức phạt,
     thời hạn, đối tượng hoặc trường hợp được hỏi.
6. Nếu không tìm thấy node phù hợp:
   trả về một mảng JSON rỗng [].
7. Không được tự tạo node_id.
8. node_id phải tồn tại chính xác trong TREE JSON.
9. `content` phải được lấy chính xác từ node tương ứng trong TREE JSON.
   Không được viết lại, tóm tắt hoặc thay đổi nội dung.
10. `reasoning` phải giải thích ngắn gọn tại sao node này liên quan
    đến câu hỏi.

ĐỊNH DẠNG OUTPUT:
Chỉ trả về JSON hợp lệ.

Format:

[
  {{
    "node_id": 123,
    "reasoning": "Giải thích ngắn gọn tại sao node này liên quan.",
    "content": "Nội dung nguyên bản của node."
  }}
]

Không được thêm:
- Markdown
- ```json
- giải thích bên ngoài JSON
- câu trả lời cho người dùng

TREE JSON:
{tree_json}

CÂU HỎI NGƯỜI DÙNG:
{query}

"""

    return prompt

In [6]:
from google.genai import types

def select_nodes(query, nodes):

    prompt = create_node_selector_prompt(
        query=query,
        nodes=nodes
    )

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json"
        )
    )

    return response.text

In [9]:
result = select_nodes(
    query=query,
    nodes=nodes
)

print(result)

[
  {
    "node_id": 855,
    "reasoning": "Khoản 2 Điều 169 quy định chi tiết về độ tuổi nghỉ hưu và lộ trình tăng tuổi nghỉ hưu đối với lao động nam trong điều kiện lao động bình thường.",
    "content": "Tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi đối với lao động nam vào năm 2028 và đủ 60 tuổi đối với lao động nữ vào năm 2035.\nKể từ năm 2021, tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường là đủ 60 tuổi 03 tháng đối với lao động nam và đủ 55 tuổi 04 tháng đối với lao động nữ; sau đó, cứ mỗi năm tăng thêm 03 tháng đối với lao động nam và 04 tháng đối với lao động nữ."
  }
]


In [10]:
selected = json.loads(result)

print(selected)

[{'node_id': 855, 'reasoning': 'Khoản 2 Điều 169 quy định chi tiết về độ tuổi nghỉ hưu và lộ trình tăng tuổi nghỉ hưu đối với lao động nam trong điều kiện lao động bình thường.', 'content': 'Tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi đối với lao động nam vào năm 2028 và đủ 60 tuổi đối với lao động nữ vào năm 2035.\nKể từ năm 2021, tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường là đủ 60 tuổi 03 tháng đối với lao động nam và đủ 55 tuổi 04 tháng đối với lao động nữ; sau đó, cứ mỗi năm tăng thêm 03 tháng đối với lao động nam và 04 tháng đối với lao động nữ.'}]


In [12]:
def fetch_selected_nodes(selected):
    selected_nodes = []

    for item in selected:
        node_id = item.get("node_id")
        if node_id not in node_index:
            raise ValueError(f"Unknown node_id: {node_id}")

        selected_nodes.append({
            "node_id": node_id,
            "reasoning": item.get("reasoning", ""),
            "content": item.get("content", node_index[node_id]["content"]),
            "node": node_index[node_id]
        })

    return selected_nodes


selected_nodes = fetch_selected_nodes(selected)
for item in selected_nodes:
    print("Node ID:", item["node_id"])
    print("Reasoning:", item["reasoning"])
    print("Content:")
    print(item["content"])
    print("=" * 80)

Node ID: 855
Reasoning: Khoản 2 Điều 169 quy định chi tiết về độ tuổi nghỉ hưu và lộ trình tăng tuổi nghỉ hưu đối với lao động nam trong điều kiện lao động bình thường.
Content:
Tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi đối với lao động nam vào năm 2028 và đủ 60 tuổi đối với lao động nữ vào năm 2035.
Kể từ năm 2021, tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường là đủ 60 tuổi 03 tháng đối với lao động nam và đủ 55 tuổi 04 tháng đối với lao động nữ; sau đó, cứ mỗi năm tăng thêm 03 tháng đối với lao động nam và 04 tháng đối với lao động nữ.


In [13]:
#Build Context for answer LLM
def build_context(selected_nodes):

    if not selected_nodes:
        return ""

    sections = []

    for item in selected_nodes:

        node = item["node"]
        lines = []

        #Hierarchy
        for p in node["path"]:

            if p["type"] == "CHUONG":
                lines.append(
                    f"Chương {p['number']} - {p['title']}"
                )

            elif p["type"] == "DIEU":
                lines.append(
                    f"Điều {p['number']} - {p['title']}"
                )

            elif p["type"] == "KHOAN":
                lines.append(
                    f"Khoản {p['number']}"
                )

            elif p["type"] == "DIEM":
                lines.append(
                    f"Điểm {p['number']}"
                )

        lines.append("")
        lines.append(node["content"])

        sections.append(
            "\n".join(lines)
        )

    separator = (
        "\n"
        + "=" * 80
        + "\n\n"
    )

    return separator.join(sections)

In [14]:
context = build_context(selected_nodes)

print(context)

Chương XII - BẢO HIỂM XÃ HỘI, BẢO HIỂM Y TẾ, BẢO HIỂM THẤT NGHIỆP
Điều 169 - Tuổi nghỉ hưu
Khoản 2

Tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi đối với lao động nam vào năm 2028 và đủ 60 tuổi đối với lao động nữ vào năm 2035.
Kể từ năm 2021, tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường là đủ 60 tuổi 03 tháng đối với lao động nam và đủ 55 tuổi 04 tháng đối với lao động nữ; sau đó, cứ mỗi năm tăng thêm 03 tháng đối với lao động nam và 04 tháng đối với lao động nữ.


In [15]:
#Answer for query LLM
from google.genai import types

def answer_question(query, context):
    if not context.strip():
        return "Không biết, hoặc thông tin không có trong dữ liệu."

    prompt = f"""
Bạn là trợ lý hỏi đáp về pháp luật Việt Nam.

Chỉ được sử dụng thông tin trong CONTEXT để trả lời.

Quy tắc:

- Không tự suy diễn.
- Không bổ sung kiến thức bên ngoài.
- Nếu Context chứa nhiều thông tin hơn câu hỏi cần biết thì chỉ trả lời câu hỏi mà context có chứa
- Trả lời ngắn gọn, rõ ràng bằng tiếng Việt.
- Đầu câu trả lời hãy thêm "Theo bộ luật lao động Việt Nam 2019" Chương nào - Điều nào - Khoản nào (nếu có, không có không cần khai báo) - Điểm nào (nếu có, không có không cần khai báo).

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2
        )
    )

    return response.text.strip()

In [16]:
answer = answer_question(
    query=query,
    context=context
)

print(query)
print()
print(answer)

Nam gới được nghỉ hưu bao nhiêu tuổi?

Theo bộ luật lao động Việt Nam 2019 Chương XII - Điều 169 - Khoản 2: 

Tuổi nghỉ hưu của lao động nam trong điều kiện lao động bình thường được quy định như sau:
- Được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi vào năm 2028.
- Kể từ năm 2021, tuổi nghỉ hưu là đủ 60 tuổi 03 tháng; sau đó, cứ mỗi năm tăng thêm 03 tháng.
